# Module 4 - Topic 5: Introduction to scikit-learn
## Live Demo Notebook

**Scenario:** A Lagos property agency wants to predict house prices from property features.

We use this dataset to demonstrate the full scikit-learn API:
- `train_test_split` - the correct order
- `StandardScaler` - fit on train, transform both
- Data leakage - the wrong way vs the right way
- `Pipeline` - preventing leakage automatically
- Evaluation metrics for regression

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# Load Lagos house price dataset
df = pd.read_csv('lagos_houses.csv')
print('Shape:', df.shape)
df.head()

---
## Part 1 - The Universal scikit-learn Pattern: fit / predict / score

In [ ]:
# The same three steps work for ANY scikit-learn algorithm
X = df[['bedrooms', 'bathrooms', 'area_sqm', 'location_tier']]
y = df['price_million_naira']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 1: CREATE
model = LinearRegression()

# Step 2: FIT
model.fit(X_train, y_train)

# Step 3: PREDICT
predictions = model.predict(X_test)

print('fit/predict/score pattern works for LinearRegression.')
print(f'Test MAE: NGN {mean_absolute_error(y_test, predictions):.1f}M')

In [ ]:
# Exact same pattern - different algorithm
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

for name, algo in [
    ('LinearRegression',    LinearRegression()),
    ('DecisionTree',        DecisionTreeRegressor(max_depth=5, random_state=42)),
    ('RandomForest',        RandomForestRegressor(n_estimators=50, random_state=42)),
]:
    algo.fit(X_train, y_train)   # same line for all three
    pred = algo.predict(X_test)  # same line for all three
    mae  = mean_absolute_error(y_test, pred)
    r2   = r2_score(y_test, pred)
    print(f'{name:<25} MAE: NGN {mae:>5.1f}M   R2: {r2:.3f}')

print()
print('The API is identical for all three - only the import and constructor change.')

---
## Part 2 - StandardScaler: Why Scaling Matters

In [ ]:
# Without scaling - feature ranges are very different
print('Feature ranges (unscaled):')
for col in X.columns:
    print(f'  {col:<20} min={X[col].min():.1f}  max={X[col].max():.1f}  range={X[col].max()-X[col].min():.1f}')

In [ ]:
# CORRECT scaling approach - fit only on training data
scaler = StandardScaler()

# Fit the scaler - learns mean and std FROM TRAINING DATA ONLY
scaler.fit(X_train)

# Transform both sets using the SAME fitted scaler
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # apply - do NOT refit

print('After scaling (mean≈0, std≈1 for each feature):')
X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print(X_train_df.describe().round(2).loc[['mean', 'std', 'min', 'max']])

---
## Part 3 - Data Leakage Demo: Wrong vs Right

In [ ]:
# WRONG: fit scaler on ALL data before splitting
scaler_wrong = StandardScaler()
X_all_scaled = scaler_wrong.fit_transform(X)   # <-- sees test data statistics!

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_all_scaled, y, test_size=0.2, random_state=42
)

wrong_model = LinearRegression()
wrong_model.fit(X_train_w, y_train_w)
wrong_mae = mean_absolute_error(y_test_w, wrong_model.predict(X_test_w))
wrong_r2  = r2_score(y_test_w, wrong_model.predict(X_test_w))

print('WRONG (data leakage):')
print(f'  MAE: NGN {wrong_mae:.1f}M   R2: {wrong_r2:.3f}')

In [ ]:
# CORRECT: split first, then fit scaler on training data only
scaler_right = StandardScaler()
scaler_right.fit(X_train)                          # fit on train only
X_tr_scaled = scaler_right.transform(X_train)
X_te_scaled = scaler_right.transform(X_test)       # apply, don't refit

right_model = LinearRegression()
right_model.fit(X_tr_scaled, y_train)
right_mae = mean_absolute_error(y_test, right_model.predict(X_te_scaled))
right_r2  = r2_score(y_test, right_model.predict(X_te_scaled))

print('CORRECT (no leakage):')
print(f'  MAE: NGN {right_mae:.1f}M   R2: {right_r2:.3f}')
print()
print('The difference may be small here - but with real high-stakes data,')
print('leakage produces models that look better than they are in production.')

---
## Part 4 - Pipeline: The Clean, Leak-Proof Solution

In [ ]:
# Build a Pipeline - it handles the correct order automatically
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression()),
])

# Fit: scaler fitted on X_train, model fitted on scaled X_train - all in one call
pipe.fit(X_train, y_train)

# Predict: X_test is scaled with the SAME fitted scaler, then model predicts
pipe_pred = pipe.predict(X_test)
pipe_mae  = mean_absolute_error(y_test, pipe_pred)
pipe_r2   = r2_score(y_test, pipe_pred)

print('Pipeline (scaler + LinearRegression):')
print(f'  MAE: NGN {pipe_mae:.1f}M')
print(f'  R2:  {pipe_r2:.3f}')
print()
print('Same result as the manual correct approach - but impossible to make a leakage mistake.')

---
## Part 5 - Interpreting Regression Metrics

In [ ]:
y_pred = pipe.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mean_price = y_test.mean()

print('=== Regression Metrics Interpretation ===')
print(f'Mean actual price: NGN {mean_price:.1f}M')
print(f'MAE:               NGN {mae:.1f}M')
print(f'MAE as % of mean:  {mae/mean_price*100:.1f}%')
print(f'R-squared:         {r2:.3f}')
print()
print(f'Interpretation:')
print(f'  On average, predictions are off by NGN {mae:.1f}M.')
print(f'  For a NGN {mean_price:.0f}M house, that is a {mae/mean_price*100:.1f}% error.')
if r2 > 0.7:
    print(f'  R2 of {r2:.3f} means the model explains {r2*100:.0f}% of price variation - reasonable.')
else:
    print(f'  R2 of {r2:.3f} - model explains {r2*100:.0f}% of variation. More features may help.')

In [ ]:
# Visualise actual vs predicted prices
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, y_pred, alpha=0.7, color='steelblue', edgecolors='white', s=80)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        color='red', linestyle='--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Price (NGN Millions)', fontsize=12)
ax.set_ylabel('Predicted Price (NGN Millions)', fontsize=12)
ax.set_title('Actual vs Predicted Lagos House Prices', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print('Points close to the diagonal line = accurate predictions.')
print('Points far from the line = large prediction errors.')

---
## Summary

| Concept | Key point |
|---|---|
| `fit/predict/score` | Universal pattern - same for all 50+ scikit-learn algorithms |
| `StandardScaler` | `fit()` on training data only; `transform()` on both sets |
| Data leakage | Fitting scaler on all data before splitting contaminates test |
| `Pipeline` | Chains scaler + model; prevents leakage automatically |
| MAE | Average error in original units (e.g. NGN millions) |
| R² | Proportion of variance explained (0–1, higher = better) |

**Next - Topic 6:** Building Your First Complete Model. End-to-end on a Nigerian sales prediction dataset - encode, split, pipeline, train, evaluate, interpret.